# Giải thích:
Đây là cách để tạo ra file sách so sánh (compare book)
Gồm 2 sách: left-book và right-book

Ghi chú: mối sách số file cần so sánh phải như nhau, và có bản map 1-1 giữa sách bên trái và sách bên phải
ví du:
```
left         right
chap1.md     chuong-1.md   
chap2.md     chuong-2.md
```

1. tạo filelist.js cho từng book left+right (3make_mucluc.ipynb)
Kết quả tạo ra file `mucluc.md` và `filelist.js` trong 
- mucluc.md dùng để làm file index hoặc bất cứ thứ gì
- filelist.js 
danh sách các text và link dùng để làm button [next][back] xem `config.ts` code phần `BOOK_NAV`
`{ text: " MN 1. CÁC NGUYÊN LÝ", link: "/kinhtrungbo/pali-vi/mn-001.md" },`


2. Tạo ra sách so sánh (l-r) 
 1. copy/paste `filelist.js` ở bước 1 vào code ở dưới (left) 
 2. copy/paste `filelist.js` ở bước 1 vào code ở dưới (right) 
 (sẽ lấy text-url theo tài liệu của right)

In [22]:
import os
import json
from slugify import slugify
import re

def merge_lists(list_left, list_right, count, left_title, right_title, output_dir, output_val, url_path, pre_slug='', title_kind="left"):
    """
    Merges two lists of dictionaries and creates a new list with combined data.

    Args:
        list_a: The first list of dictionaries.
        list_b: The second list of dictionaries.
        count: The number of elements to merge.
        left_title:  The title for the 'left' side (from list_b).
        right_title: The title for the 'right' side (from list_a).
        dest_folder: save mucluc.md path
        pre_slug: dùng dnc='kinh trường bộ compare'

    Returns:
        A new list of dictionaries with the merged data.
    """

    merged_list = []
    title_item = list_left if title_kind == "left" else list_right

    def slug_link(item):
        return f"{pre_slug}{os.path.basename(item['link'])}".replace('.md', '')[2:]

    def make_slug(text: str) -> str:

        match = re.match(r'^[A-Z]+\s+(\d+)\.\s+(.+)$', text.strip())
        if not match:
            raise ValueError(f"Không đúng định dạng: {text!r}")

        num   = int(match.group(1))          # 1  →  001
        title = match.group(2)               # "KINH LƯỚI TRỜI"

        return f"{pre_slug}{num:03d}-{slugify(title)}"

    for i in range(min(count, len(list_left), len(list_right))):  # Iterate up to the shortest list length or 'count'
        left_item = list_left[i]
        right_item = list_right[i]

        slug = make_slug(title_item[i]['text'])
        #slug = slug_link(left_item) # os.path.basename(a_item['link']).replace('.md', '')


        title = title_item[i]['text']


        # 3. & 4. Extract links
        right_link = left_item['link']
        left_link = right_item['link']

        # 5. & 6.  Use input titles
        # (already have them as function parameters)

        # 7. notePath (always empty in this example)
        note_path = ""

        # 8. backlink and nextlink
        backlink = False if i == 0 else {
            "text": title_item[i - 1]['text'],  # Text from the *previous* item in list_a
            "link": make_slug(title_item[i-1]['text'])  # f"{url_path}{os.path.basename(list_a[i-1]['link']).replace('.md', '')}" # link from previous item
        }


        nextlink = False if i == min(count, len(list_left), len(list_right)) - 1 else {
            "text": title_item[i + 1]['text'],  # Text from the *next* item in list_a
            "link": make_slug(title_item[i+1]['text'])  #f"{url_path}{os.path.basename(list_a[i+1]['link']).replace('.md', '')}"# link from previous item

        }

        merged_item = {
            "params": {
                "slug": slug,
                "data": {
                    "title": title,
                    "left": left_link,
                    "right": right_link,
                    "leftTitle": left_title,
                    "rightTitle": right_title,
                    "notePath": note_path,
                    "backlink": backlink,
                    "nextlink": nextlink
                }
            }
        }

        merged_list.append(merged_item)

    # Create the output directory if it doesn't exist
    # output_dir = os.path.dirname(output_file)
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Write the merged list to the JavaScript file
    tmc_file = output_dir+"tmc.js"
    with open(tmc_file, 'w', encoding='utf-8') as f:
        f.write(f"const {output_val} = ")
        json.dump(merged_list, f, indent=2, ensure_ascii=False)  # Use json.dump
        f.write(";\n")
        f.write(f"export default {output_val};\n")

    print(f"Created tmc.js file at {tmc_file} file")

    # Generate the Markdown table of contents (mucluc.md)
    md_toc = ""
    for item in merged_list:
        md_toc += f'- [{item["params"]["data"]["title"]}]({url_path}{item["params"]["slug"]})\n'

    # Write the Markdown TOC to mucluc.md
    mucluc_path = os.path.join(output_dir, "mucluc.md")
    try:
        with open(mucluc_path, 'w', encoding='utf-8') as outfile:
            outfile.write(md_toc)
        print(f"Successfully generated mucluc.md in {mucluc_path}")
    except Exception as e:
        print(f"Error writing to mucluc.md: {e}")

    return merged_list

In [24]:
# step 1 - MAKE COMPARE FILE
# copy list from filelist.js create in make_filelist
text = "text"
link = "link"
# 23
left_panel =  [
         { text: "DN 1. KINH LƯỚI TRỜI", link: "/kinhtruongbo/pali-vi/dn-001.md" },
  { text: "DN 2. KINH VỀ THÀNH QUẢ CỦA NGƯỜI TU HÀNH", link: "/kinhtruongbo/pali-vi/dn-002.md" },
  { text: "DN 3. KINH AMBAṬṬHA", link: "/kinhtruongbo/pali-vi/dn-003.md" },
  { text: "DN 4. KINH SOṆADAṆḌA", link: "/kinhtruongbo/pali-vi/dn-004.md" },
  { text: "DN 5. KINH KŪṬADANTA", link: "/kinhtruongbo/pali-vi/dn-005.md" },
  { text: "DN 6. KINH MAHĀLI", link: "/kinhtruongbo/pali-vi/dn-006.md" },
  { text: "DN 7. KINH JĀLIYA", link: "/kinhtruongbo/pali-vi/dn-007.md" },
  { text: "DN 8. KINH TIẾNG GẦM SƯ TỬ", link: "/kinhtruongbo/pali-vi/dn-008.md" },
  { text: "DN 9. KINH POṬṬHAPĀDA", link: "/kinhtruongbo/pali-vi/dn-009.md" },
  { text: "DN 10. KINH SUBHA", link: "/kinhtruongbo/pali-vi/dn-010.md" },
  { text: "DN 11. KINH KEVAṬṬA", link: "/kinhtruongbo/pali-vi/dn-011.md" },
  { text: "DN 12. KINH LOHICCA", link: "/kinhtruongbo/pali-vi/dn-012.md" },
  { text: "DN 13. KINH TEVIJJA (Kinh Ba Minh)", link: "/kinhtruongbo/pali-vi/dn-013.md" },
  { text: "DN 14. KINH DÀI VỀ NGUỒN GỐC", link: "/kinhtruongbo/pali-vi/dn-014.md" },
  { text: "DN 15. KINH DÀI VỀ QUAN HỆ PHỤ THUỘC", link: "/kinhtruongbo/pali-vi/dn-015.md" },
  { text: "DN 16. KINH DÀI VỀ BÁT-NIẾT-BÀN", link: "/kinhtruongbo/pali-vi/dn-016.md" },
  { text: "DN 17. KINH ĐẠI THIỆN KIẾN VƯƠNG", link: "/kinhtruongbo/pali-vi/dn-017.md" },
  { text: "DN 18. KINH JANAVASABHA", link: "/kinhtruongbo/pali-vi/dn-018.md" },
  { text: "DN 19. KINH DÀI VỀ GOVINDA", link: "/kinhtruongbo/pali-vi/dn-019.md" },
  { text: "DN 20. KINH ĐẠI HỘI", link: "/kinhtruongbo/pali-vi/dn-020.md" },
  { text: "DN 21. KINH SAKKAPAÑHA (ĐẾ THÍCH SỞ VẤN)", link: "/kinhtruongbo/pali-vi/dn-021.md" },
  { text: "DN 22. KINH DÀI VỀ BỐN NƠI CHÚ TÂM", link: "/kinhtruongbo/pali-vi/dn-022.md" },
  { text: "DN 23. KINH Pāyāsi", link: "/kinhtruongbo/pali-vi/dn-023.md" },
  { text: "DN 24. KINH PĀTHIKA", link: "/kinhtruongbo/pali-vi/dn-024.md" },
  { text: "DN 25. KINH UDUMBARIKA", link: "/kinhtruongbo/pali-vi/dn-025.md" },
  { text: "DN 26. KINH VUA CHUYỂN LUÂN", link: "/kinhtruongbo/pali-vi/dn-026.md" },
  { text: "DN 27. KINH VỀ NGUỒN GỐC TỐI SƠ", link: "/kinhtruongbo/pali-vi/dn-027.md" },
  { text: "DN 28. KINH NIỀM TIN TRONG SÁNG", link: "/kinhtruongbo/pali-vi/dn-028.md" },
  { text: "DN 29. KINH RÕ RÀNG VÀ MẠCH LẠC", link: "/kinhtruongbo/pali-vi/dn-029.md" },
  { text: "DN 30. KINH ĐẶC ĐIỂM", link: "/kinhtruongbo/pali-vi/dn-030.md" },
  { text: "DN 31. KINH SIṄGĀLA", link: "/kinhtruongbo/pali-vi/dn-031.md" },
  { text: "DN 32. KINH ĀṬĀNĀṬIYA", link: "/kinhtruongbo/pali-vi/dn-032.md" },
  { text: "DN 33. KINH TỤNG ĐỌC CÙNG NHAU", link: "/kinhtruongbo/pali-vi/dn-033.md" },
  { text: "DN 34. KINH ĐẾN MƯỜI", link: "/kinhtruongbo/pali-vi/dn-034.md" },
]

right_panel = [
 { text: "1. KINH PHẠM VÕNG", link: "/kinhtruongbo/thichminhchau/dn-001-kinh-pham-vong.md" },
  { text: "2. KINH SA MÔN QỦA", link: "/kinhtruongbo/thichminhchau/dn-002-kinh-sa-mon-qua.md" },
  { text: "3. KINH AMBATTHA (A-MA-TRÚ)", link: "/kinhtruongbo/thichminhchau/dn-003-kinh-ambattha-a-ma-tru.md" },
  { text: "4. KINH SONADANDA (CHỦNG ÐỨC)", link: "/kinhtruongbo/thichminhchau/dn-004-kinh-sonadanda-chung-duc.md" },
  { text: "5. KINH KÙTADANTA (CỨU-LA-ÐÀN-ÐẦU)", link: "/kinhtruongbo/thichminhchau/dn-005-kinh-kutadanta-cuu-la-dan-dau.md" },
  { text: "6. KINH MAHÀLI", link: "/kinhtruongbo/thichminhchau/dn-006-kinh-mahali.md" },
  { text: "7. KINH JÀLIYA", link: "/kinhtruongbo/thichminhchau/dn-007-kinh-jaliya.md" },
  { text: "8. KINH CA-DIẾP SƯ TỬ HỐNG", link: "/kinhtruongbo/thichminhchau/dn-008-kinh-ca-diep-su-tu-hong.md" },
  { text: "9. KINH POTTHAPÀDA (BỐ-SÁ-BÀ-LÂU)", link: "/kinhtruongbo/thichminhchau/dn-009-kinh-potthapada-bo-sa-ba-lau.md" },
  { text: "10. KINH SUBHA (TU-BÀ)", link: "/kinhtruongbo/thichminhchau/dn-010-kinh-subha-tu-ba.md" },
  { text: "11. KINH KEVADDHA (KIÊN CỐ)", link: "/kinhtruongbo/thichminhchau/dn-011-kinh-kevaddha-kien-co.md" },
  { text: "12. KINH LOHICCA (LÔ-HI-GIA)", link: "/kinhtruongbo/thichminhchau/dn-012-kinh-lohicca-lo-hi-gia.md" },
  { text: "13. KINH TEVIJJA (TAM MINH)", link: "/kinhtruongbo/thichminhchau/dn-013-kinh-tevijja-tam-minh.md" },
  { text: "14. KINH ÐẠI BỔN", link: "/kinhtruongbo/thichminhchau/dn-014-kinh-dai-bon.md" },
  { text: "15. KINH ÐẠI DUYÊN", link: "/kinhtruongbo/thichminhchau/dn-015-kinh-dai-duyen.md" },
  { text: "16. KINH ÐẠI BÁT NIẾT BÀN", link: "/kinhtruongbo/thichminhchau/dn-016-kinh-dai-bat-niet-ban.md" },
  { text: "17. KINH ÐẠI THIỆN KIẾN VƯƠNG", link: "/kinhtruongbo/thichminhchau/dn-017-kinh-dai-thien-kien-vuong.md" },
  { text: "18. KINH XA-NI-SA", link: "/kinhtruongbo/thichminhchau/dn-018-kinh-xa-ni-sa.md" },
  { text: "19. KINH ÐẠI ÐIỂN TÔN", link: "/kinhtruongbo/thichminhchau/dn-019-kinh-dai-dien-ton.md" },
  { text: "20. KINH ÐẠI HỘI", link: "/kinhtruongbo/thichminhchau/dn-020-kinh-dai-hoi.md" },
  { text: "21. KINH ÐẾ-THÍCH SỞ VẤN", link: "/kinhtruongbo/thichminhchau/dn-021-kinh-de-thich-so-van.md" },
  { text: "22. KINH ÐẠI NIỆM XỨ", link: "/kinhtruongbo/thichminhchau/dn-022-kinh-dai-niem-xu.md" },
  { text: "23. KINH TỆ-TÚC", link: "/kinhtruongbo/thichminhchau/dn-023-kinh-te-tuc.md" },
  { text: "24. KINH BA-LÊ", link: "/kinhtruongbo/thichminhchau/dn-024-kinh-ba-le.md" },
  { text: "25. KINH ƯU-ÐÀM-BÀ-LA SƯ TỬ HỐNG", link: "/kinhtruongbo/thichminhchau/dn-025-kinh-uu-dam-ba-la-su-tu-hong.md" },
  { text: "26. KINH CHUYỂN LUÂN THÁNH VƯƠNG SƯ TỬ HỐNG", link: "/kinhtruongbo/thichminhchau/dn-026-kinh-chuyen-luan-thanh-vuong-su-tu-hong.md" },
  { text: "27. KINH KHỞI THẾ NHÂN BỔN", link: "/kinhtruongbo/thichminhchau/dn-027-kinh-khoi-the-nhan-bon.md" },
  { text: "28. KINH TỰ HOAN HỶ", link: "/kinhtruongbo/thichminhchau/dn-028-kinh-tu-hoan-hy.md" },
  { text: "29. KINH THANH TỊNH", link: "/kinhtruongbo/thichminhchau/dn-029-kinh-thanh-tinh.md" },
  { text: "30. KINH TƯỚNG", link: "/kinhtruongbo/thichminhchau/dn-030-kinh-tuong.md" },
  { text: "31. KINH GIÁO THỌ THI-CA-LA-VIỆT", link: "/kinhtruongbo/thichminhchau/dn-031-kinh-giao-tho-thi-ca-la-viet.md" },
  { text: "32. KINH A-SÁ-NANG-CHI", link: "/kinhtruongbo/thichminhchau/dn-032-kinh-a-sa-nang-chi.md" },
  { text: "33. KINH PHÚNG TỤNG", link: "/kinhtruongbo/thichminhchau/dn-033-kinh-phung-tung.md" },
  { text: "34. KINH THẬP THƯỢNG", link: "/kinhtruongbo/thichminhchau/dn-034-kinh-thap-thuong.md" },
]
count = 500
left_title = "Pali (vi)"
right_title = "HT Thích Minh Châu"

alang = "vi"
kinh="kinhtruongbo"
author1="pali"
author2="tmc"
save_to_dir = f"../docs/{kinh}/c-{author1}-{author2}-{alang}/"  # result file saved directory
print_author_export = f"{author1}_{author2}_{alang}"  # Example variable name
print_url_path = f"/{kinh}/c-{author1}-{author2}-{alang}/"
pre_slug = "dnc-"
merged = merge_lists(right_panel, left_panel, count, left_title, right_title, save_to_dir, print_author_export, print_url_path, pre_slug, title_kind="right")
# import json
# print(json.dumps(merged, indent=2, ensure_ascii=False)) # use json for pretty printing

Created tmc.js file at ../docs/kinhtruongbo/c-pali-tmc-vi/tmc.js file
Successfully generated mucluc.md in ../docs/kinhtruongbo/c-pali-tmc-vi/mucluc.md
